In [ ]:
import pandas as pd

## Read gas

In [ ]:
# Paths are relative to this notebook. Run the case in foam_template first
# (or point to temp/<test name>/ after a pytest run).
T_probes = pd.read_csv('foam_template/postProcessing/probes/0/T',skiprows=4,sep='\s+',usecols=[0,1,2,3])
T_probes = T_probes.set_axis(['Time','Position 1','Position 2','Position 3'],axis=1)
T_probes

## Read particles

In [ ]:
import pyvista as pv

In [ ]:
# Paths are relative to this notebook. Run the case in foam_template first
# (or point to temp/<test name>/ after a pytest run).
import json
import os 
import numpy as np

prt_dfs = {}
for cloud_path, cloud_name in zip(['writeCarbCloud'],['carbCloud']):
    path = os.path.join('foam_template/postProcessing', cloud_path)
    with open(os.path.join(path,cloud_name + '.vtp.series')) as f:
        d = json.load(f)

    time_file_dict = {}
    times = []
    files = []
    for di in d['files']:
        times.append(di['time'])
        files.append(os.path.join(path,di['name']))

    sorting = np.argsort(np.array(times))
    files = list(np.array(files)[sorting])

    print('Reading',cloud_name)
    data_list = []
    for file in files:
        vtk = pv.read(file)
        data_list.append(np.concatenate((vtk['T'][:,np.newaxis],vtk['origId'][:,np.newaxis],vtk['TimeValue']*np.ones(vtk.n_points)[:,np.newaxis]),axis=1))
    data = np.concatenate(tuple(data_list))
    prt_dfs[cloud_name] = pd.DataFrame({'T':data[:,0],'origId':data[:,1],'Time':data[:,2]})

In [ ]:
prt_dfs

In [ ]:
for cloud_name in ['carbCloud']:
    prt_dfs[cloud_name] = prt_dfs[cloud_name].groupby('origId')

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
prt_dfs['carbCloud'].groups.keys()

## Visualize

In [ ]:
fig, ax = plt.subplots(1,1)

solid_lines = []
for carb_id, pos in zip(prt_dfs['carbCloud'].groups.keys(),['Position 1','Position 2','Position 3']):
    line = ax.plot(T_probes['Time'],T_probes[pos],label=pos)[0]
    solid_lines.append(line)
    prt = prt_dfs['carbCloud'].get_group(carb_id)
    ax.plot(prt['Time'],prt['T'],c=line.get_color(),linestyle='--')
    ax.grid()

ax.set_ylabel('$T$ (K)')
ax.set_xlabel('$t$ (s)')

# Create solid, dashed lines in legend (dashed per-position lines above are
# left unlabeled so they don't each get their own legend entry)
import matplotlib.lines as mlines
# Custom legend handles
solid_line = mlines.Line2D([], [], color="black", linestyle="-", label="Probe")
dashed_line = mlines.Line2D([], [], color="black", linestyle="--", label="Carbonaceous Cloud")
ax.legend(handles=solid_lines+[solid_line,dashed_line])

fig.savefig('reference/figures/reference_profiles.png')

## Write reference data

In [ ]:
joint_df = pd.DataFrame({'Time':T_probes['Time'],'Probe 1': T_probes['Position 1'], 'Probe 2': T_probes['Position 2'], 'Probe 3': T_probes['Position 3']})
# joint_df = joint_df.iloc[:-1]
joint_df

In [ ]:
for cloud_name, grouped_df in prt_dfs.items():
    for i, id in enumerate(grouped_df.groups.keys()):
        tmp_df = grouped_df.get_group(id)[['T']].rename(columns={'T':cloud_name + ' Position ' + str(i+1)}).iloc[1:].reset_index(drop=True)
        joint_df = pd.concat([joint_df,tmp_df],axis=1)
joint_df

In [ ]:
joint_df.to_csv('reference/foam/reference_data.csv')